In [ ]:
import pandas as pd
import requests
import json
import matplotlib.pyplot as plt

import matplotlib.patches as mpatches
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"Pandas version:{pd.__version__}")
print(f"Requests version:{requests.__version__}")
print(f"Loaded at:{datetime.now().strftime('%d/%m/%Y, %H:%M:%S')}")


Libraries imported successfully
Pandas version:2.2.2
Requests version:2.32.4
Loaded at:27/05/2026, 04:28:10


In [ ]:
raw_df=pd.read_csv('/content/drive/MyDrive/codebooster-datasets/messy_sales_data.csv')
print(f"Rows:{raw_df.shape[0]}")
print(f"Columns:{raw_df.shape[1]}")
print(raw_df.shape)
print(raw_df.columns.tolist())
print("First 5 rows")
raw_df.head()

Rows:30
Columns:9
(30, 9)
['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']
First 5 rows


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [ ]:
print("\nmissing values count")
print(raw_df.isnull().sum())

print("\nduplicate rows")
print(raw_df.duplicated().sum())

print("\ndata types")
print(raw_df.dtypes)

print("\nunique categories")
print(raw_df['category'].dropna().unique().tolist())

print("\nsample order dates")
print(raw_df['order_date'].unique()[:8].tolist())

print("\nsample names")
print(raw_df['customer_name'].dropna().unique()[:8].tolist())


missing values count
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

duplicate rows
0

data types
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

unique categories
['Electronics', 'Accessories']

sample order dates
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15']

sample names
['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'Ananya Das', 'Vikram Iyer']


In [ ]:
df=raw_df.copy()

print(df.shape)
print("raw df is untouched.we can always go with: df=raw_df.copy()")

(30, 9)
raw df is untouched.we can always go with: df=raw_df.copy()


In [ ]:
print("before cleaning:")
print(df.isnull().sum().sum())



before cleaning:
0


In [ ]:
df['customer_name'].fillna('Unknown Customer',inplace=True)

print(df['customer_name'].isnull().sum())

#===========================================

median_qty=df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)

print(df['quantity'].isnull().sum())

#============================================

df['category'].fillna('Unknown Category',inplace=True)

print(df['category'].isnull().sum())

#===============================================
df['product'].fillna('Unknown product',inplace=True)

print(df['product'].isnull().sum())


print("after cleaning:")
print(df.isnull().sum().sum())


0
0
0
0
after cleaning:
0


In [ ]:
dup=raw_df.copy()
print("before cleaning:")
print(dup.isnull().sum().sum())

before cleaning:
7


In [ ]:
dup['customer_name'].fillna('Unknown Customer',inplace=True)

median_qty=df['quantity'].median()
dup['quantity'].fillna(median_qty,inplace=True)

dup['category'].fillna('Unknown Category',inplace=True)

dup['product'].fillna('Unknown product',inplace=True)

print("after cleaning:")
print(dup.isnull().sum().sum())


after cleaning:
0


In [ ]:
print(dup.duplicated().sum())

0


In [ ]:
len(dup)

30

In [ ]:
duplicate_mask=dup.duplicated(keep=False)
print("\nthe duplicate rows ")
print(dup[duplicate_mask][['order_id','customer_name','product','order_date']].to_string(index=True))


dup.drop_duplicates(inplace=True)
dup.reset_index(drop=True,inplace=True)

print(f"\nRows After deduplication:{len(dup)}")
print(f"Rows removed:{len(raw_df)-len(dup)}")


the duplicate rows 
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

Rows After deduplication:30
Rows removed:0


# MIXED DATE FORMATS

In [ ]:
print("Sample dates before parrsing:")
print(df['order_date'].head(10).tolist())

dup['order_date']=pd.to_datetime(dup['order_date'],dayfirst=False,errors='coerce')

nat_mask=df['order_date'].isnull()
dup.loc[nat_mask,'order_date']=pd.to_datetime(raw_df.loc[nat_mask,'order_date'],dayfirst=True,errors='coerce')

nat_count=df['order_date'].isnull().sum()
print(f"\nUnparseable dates remaining (NaT):{nat_count}")


dup['year']=dup['order_date'].dt.year #2024
dup['month']=dup['order_date'].dt.month #1,2,3...
dup['month_name']=dup['order_date'].dt.strftime('%B') #January,February,March..
dup['day_name']=dup['order_date'].dt.strftime('%A') #Monday,Tuesday,Wednesday


print("Sample dates after parrsing:")
print(dup[['order_date','year','month','month_name','day_name']].head(8).to_string())


Sample dates before parrsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15', '2024-01-15']

Unparseable dates remaining (NaT):0
Sample dates after parrsing:
  order_date    year  month month_name   day_name
0 2024-01-05  2024.0    1.0    January     Friday
1 2024-01-07  2024.0    1.0    January     Sunday
2 2024-01-08  2024.0    1.0    January     Monday
3 2024-01-10  2024.0    1.0    January  Wednesday
4 2024-01-05  2024.0    1.0    January     Friday
5        NaT     NaN    NaN        NaN        NaN
6 2024-01-12  2024.0    1.0    January     Friday
7 2024-01-13  2024.0    1.0    January   Saturday


In [ ]:
print("\nName before standardization")
print(dup['customer_name'].unique()[:8].tolist())

dup['customer_name']=dup['customer_name'].str.strip().str.title()

print("\nNames after standardization")
print(dup['customer_name'].unique()[:8].tolist())



Name before standardization
['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'Unknown Customer', 'Ananya Das']

Names after standardization
['Ramesh Kumar', 'Priya Nair', 'Amit Verma', 'Sunita Patel', 'Kiran Mehta', 'Deepak Singh', 'Unknown Customer', 'Ananya Das']


In [ ]:
import pandas as pd # Ensure pandas is imported if not already, or remove if guaranteed by an earlier cell

# Define raw_df locally within this cell to prevent NameError if previous cells are not run
raw_df=pd.read_csv('/content/drive/MyDrive/codebooster-datasets/messy_sales_data.csv')

# Re-create dup and apply all cleaning steps for the validation report
dup = raw_df.copy()

# Fill missing values (from cell v5AQot0X5opf)
dup['customer_name'] = dup['customer_name'].fillna('Unknown Customer')
median_qty = dup['quantity'].median()
dup['quantity'] = dup['quantity'].fillna(median_qty)
dup['category'] = dup['category'].fillna('Unknown Category')
dup['product'] = dup['product'].fillna('Unknown product')

# Drop duplicates (from cell yqIO5N7DGajP)
dup.drop_duplicates(inplace=True)
dup.reset_index(drop=True, inplace=True)

# Process order_date and extract features (from cell dKaMN6K8IgQL)
dup['order_date'] = pd.to_datetime(dup['order_date'], dayfirst=False, errors='coerce')
nat_mask = dup['order_date'].isnull()
dup.loc[nat_mask, 'order_date'] = pd.to_datetime(raw_df.loc[nat_mask, 'order_date'], dayfirst=True, errors='coerce')
dup['year'] = dup['order_date'].dt.year
dup['month'] = dup['order_date'].dt.month
dup['month_name'] = dup['order_date'].dt.strftime('%B')
dup['day_name'] = dup['order_date'].dt.strftime('%A')

# Standardize customer_name (from cell I0r78xT8LwXM)
dup['customer_name'] = dup['customer_name'].str.strip().str.title()

# Calculate revenue (New step required for validation report)
dup['revenue'] = dup['quantity'] * dup['unit_price']

print("dup DataFrame prepared with all cleaning steps and revenue column.")

dup DataFrame prepared with all cleaning steps and revenue column.


In [ ]:
# ============================================================
# CELL 11 — Post-Cleaning Validation Report
# ============================================================


# Calculate the data quality score
# We check 5 things: no missing values, no duplicates, no date nulls, no revenue nulls
# Each passing check contributes 20 points (5 checks × 20 = 100)
missing_count   = dup.isnull().sum().sum()
duplicate_count = dup.duplicated().sum()
date_nulls      = dup['order_date'].isnull().sum()
revenue_nulls   = dup['revenue'].isnull().sum()


checks_passed   = sum([
    missing_count   == 0,   # 20 points
    duplicate_count == 0,   # 20 points
    date_nulls      == 0,   # 20 points
    revenue_nulls   == 0,   # 20 points
    len(dup)         > 0     # 20 points (dataset is not empty)
])
quality_score = checks_passed * 20


# ── Print the report ─────────────────────────────────────────
print("=" * 55)
print("  POST-CLEANING VALIDATION REPORT")
print("=" * 55)
print(f"  Original rows   : {len(raw_df)}")
print(f"  Cleaned rows    : {len(dup)}")
print(f"  Rows removed    : {len(raw_df) - len(dup)} (duplicates)")
print(f"  Missing values  : {missing_count}")
print(f"  Duplicates      : {duplicate_count}")
print(f"  Date nulls      : {date_nulls}")
print(f"  Revenue nulls   : {revenue_nulls}")
print(f"  Columns total   : {len(dup.columns)}")
print("=" * 55)
print(f"  DATA QUALITY SCORE : {quality_score}/100")
print(f"  DATA IS CLEAN      : {quality_score == 100}")
print("=" * 55)


# ── Actionable Debugging Suggestions ─────────────────────────
if missing_count > 0:
    print("\n  ACTION REQUIRED: Missing values detected.")
    print("  → Use df['column'].fillna(value, inplace=True)")
    print("  → For numbers: fillna(df['column'].median())")
    print("  → For text   : fillna('Unknown')")


if duplicate_count > 0:
    print("\n  ACTION REQUIRED: Duplicate rows detected.")
    print("  → Use df.drop_duplicates(inplace=True)")


if date_nulls > 0:
    print("\n  ACTION REQUIRED: Unparseable dates found.")
    print("  → Check for unusual date formats in the raw data")
    print("  → Use pd.to_datetime(col, dayfirst=True, errors='coerce')")


if quality_score == 100:
    print("\n  All checks passed. Data is ready for analysis.")

  POST-CLEANING VALIDATION REPORT
  Original rows   : 30
  Cleaned rows    : 30
  Rows removed    : 0 (duplicates)
  Missing values  : 0
  Duplicates      : 0
  Date nulls      : 0
  Revenue nulls   : 0
  Columns total   : 14
  DATA QUALITY SCORE : 100/100
  DATA IS CLEAN      : True

  All checks passed. Data is ready for analysis.


In [ ]:
output_filename='clean_sales_data.csv'
dup.to_csv(output_filename,index=False)
print(f"Cleaned data saved to :{output_filename}")
print(f"Final dataset:{dup.shape[0]} rows x {dup.shape[1]}columns")

Cleaned data saved to :clean_sales_data.csv
Final dataset:30 rows x 14columns


# Job analytics ETL Project(SerpAPI)

In [ ]:
import requests

In [ ]:
SERP_API_KEY = 'e3fbe4dc1a0be0fa382f057c9e8806aa4058c66db4e4415df93a00f974db40a7'
SERP_URL = 'https://serpapi.com/search.json'

SEARCH_QUERY = 'Full Stack Developer India'
print(f"SerpAPI Key  : {'Set (live data)' if SERP_API_KEY != 'YOUR_SERPAPI_KEY_HERE' else 'Not set (fallback data will be used)'}")

print(f"Search query:{SEARCH_QUERY}")

SerpAPI Key  : Set (live data)
Search query:Full Stack Developer India


In [ ]:
def fetch_jobs(query, api_key, num_pages=2):
    """
    Fetches job listings from Google Jobs via SerpAPI.


    Parameters:
        query    (str) : The job search query (e.g. 'Data Engineer India')
        api_key  (str) : Your SerpAPI key
        num_pages(int) : Number of result pages to fetch (default: 2)


    Returns:
        list : A list of job dictionaries
    """
    all_jobs = []


    for page in range(num_pages):
        # API pagination: 'start' tells the API which result to start from
        # Page 0: results 0-9, Page 1: results 10-19, etc.
        params = {
            'engine'    : 'google_jobs',  # Use the Google Jobs search engine
            'q'         : query,
            'api_key'   : api_key,
            'hl'        : 'en',           # Language: English
            'start'     : page * 10       # Pagination offset
        }


        try:
            response = requests.get(SERP_URL, params=params, timeout=15)


            if response.status_code == 200:
                data = response.json()


                # 'jobs_results' is the key in the JSON that holds the job listings
                jobs = data.get('jobs_results', [])


                for job in jobs:
                    # Extract and normalize each job's fields
                    # .get('key', 'default') returns the value if the key exists,
                    # or 'default' if it does not — prevents KeyError crashes
                    all_jobs.append({
                        'title'      : job.get('title', 'Unknown Title'),
                        'company'    : job.get('company_name', 'Unknown Company'),
                        'location'   : job.get('location', 'Unknown Location'),
                        'posted'     : job.get('detected_extensions', {}).get('posted_at', 'Unknown'),
                        'salary'     : job.get('detected_extensions', {}).get('salary', 'Not Disclosed'),
                        'job_type'   : job.get('detected_extensions', {}).get('schedule_type', 'Not Specified'),
                        'description': job.get('description', '')[:300]  # First 300 characters only
                    })


                print(f"  Page {page + 1}: fetched {len(jobs)} jobs")
            else:
                print(f"  Page {page + 1}: API error {response.status_code}")


        except Exception as e:
            print(f"  Page {page + 1}: error — {e}")


    return all_jobs


# ── Actually call the function ────────────────────────────────
job_records = []


if SERP_API_KEY != 'SERP_API_KEY ':
    print(f"Fetching job listings for: '{SEARCH_QUERY}'")
    job_records = fetch_jobs(SEARCH_QUERY, SERP_API_KEY)
    print(f"Total jobs fetched: {len(job_records)}")
else:
    print("No SerpAPI key provided — fallback job data will be loaded next.")

Fetching job listings for: 'Full Stack Developer India'
  Page 1: fetched 10 jobs
  Page 2: fetched 10 jobs
Total jobs fetched: 20


In [ ]:
output_filename='clean_full_stack_data.csv'

# Convert job_records (list of dicts) to a DataFrame
job_df = pd.DataFrame(job_records)

# Save the DataFrame to CSV
job_df.to_csv(output_filename,index=False)

print(f"Cleaned job data saved to: {output_filename}")
print(f"Final dataset: {job_df.shape[0]} rows x {job_df.shape[1]} columns")

Cleaned job data saved to: clean_full_stack_data.csv
Final dataset: 20 rows x 7 columns
